# wmh-multisite — mesure de la vitesse d'entraînement nnU-Net (Kaggle)

But : estimer **précisément** la durée d'une epoch de `3d_fullres` sur **notre** jeu `Dataset001_WMH` (FLAIR + T1 corrigées par N4, fold 0 stratifié), pour fixer le nombre d'epochs de l'entraînement (étape 5.2).

Le test de faisabilité donnait ~194 s par epoch (images `pre/` des organisateurs, 1 GPU, 3 processus d'augmentation, `torch.compile` actif). Kaggle n'offre que **4 cœurs CPU** : l'augmentation de données (sur CPU) peut limiter la vitesse plus que le GPU. On mesure donc 4 réglages :

| Essai | GPU | Processus d'augmentation (`nnUNet_n_proc_DA`) | `torch.compile` | Question |
|---|---|---|---|---|
| A | 1 | 3 | oui | référence (= test de faisabilité) |
| B | 1 | 4 | oui | un processus de plus aide-t-il ? |
| C | 1 | 4 | non | la compilation fait-elle gagner du temps ? |
| D | 2 | 4 | oui | deux T4 en parallèle (DDP) accélèrent-elles ? |

Chaque essai lance l'entraînement réel (même plan, même fold) et est **arrêté dès que 4 epochs sont mesurées** (la 1re, qui inclut la compilation, est exclue de la médiane). Pendant chaque essai, l'utilisation du GPU et du CPU est échantillonnée : un GPU peu utilisé avec un CPU saturé signifie que **l'augmentation de données est le goulot d'étranglement**.

## Instructions

1. **Add Input** (panneau de droite) : ajouter le dataset privé contenant `Dataset001_WMH` (déposé depuis `data/nnunet/Dataset001_WMH.zip`).
2. **Settings** : Accelerator **GPU T4 x2** (indispensable pour l'essai D), Internet **On**.
3. **Run All**. Durée estimée : 1 h à 1 h 15 de quota GPU.
4. Copier dans la conversation le **bloc JSON** affiché par la dernière cellule (aussi enregistré dans `/kaggle/working/epoch_benchmark.json`).

In [ ]:
# 1. Environment and dataset discovery
import glob, json, os, re, shutil, subprocess, threading, time, zipfile
from pathlib import Path
import psutil, torch

REPORT = {"started": time.strftime("%Y-%m-%d %H:%M:%S")}
REPORT["env"] = {"torch": torch.__version__, "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                 "cpus": os.cpu_count(), "ram_gb": round(psutil.virtual_memory().total / 1e9, 1)}
print(REPORT["env"])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"

found = glob.glob("/kaggle/input/**/Dataset001_WMH/dataset.json", recursive=True)
if not found:   # Kaggle did not unzip the archive: do it
    zips = glob.glob("/kaggle/input/**/Dataset001_WMH.zip", recursive=True)
    assert zips, "Dataset001_WMH not found in /kaggle/input: add the private dataset with 'Add Input'"
    zipfile.ZipFile(zips[0]).extractall("/tmp/raw")
    found = glob.glob("/tmp/raw/**/Dataset001_WMH/dataset.json", recursive=True)
SRC = Path(found[0]).parent

# Diagnose what Kaggle actually stored: it may decompress every .gz file of an uploaded archive,
# turning WMH_000_0000.nii.gz into WMH_000_0000.nii, while dataset.json still says ".nii.gz".
def count(folder):
    files = [p.name for p in (SRC / folder).iterdir()]
    return {"total": len(files), ".nii.gz": sum(f.endswith(".nii.gz") for f in files),
            ".nii": sum(f.endswith(".nii") for f in files), "examples": sorted(files)[:3]}
REPORT["input_files"] = {f: count(f) for f in ("imagesTr", "labelsTr")}
print("source:", SRC); print(json.dumps(REPORT["input_files"], indent=1))
ending = ".nii.gz" if REPORT["input_files"]["labelsTr"][".nii.gz"] else ".nii"
assert REPORT["input_files"]["labelsTr"][ending] == 60 and REPORT["input_files"]["imagesTr"][ending] == 120,     "unexpected content: see the counts above"

# Writable copy of the dataset in /tmp, made of symbolic links (no data copied), with the real file ending.
RAW_DATASET = Path("/tmp/raw/Dataset001_WMH")
shutil.rmtree(RAW_DATASET, ignore_errors=True)
for folder in ("imagesTr", "labelsTr"):
    (RAW_DATASET / folder).mkdir(parents=True)
    for f in (SRC / folder).iterdir():
        if f.name.endswith(ending):
            (RAW_DATASET / folder / f.name).symlink_to(f)
ds = json.load(open(SRC / "dataset.json"))
ds["file_ending"] = ending
json.dump(ds, open(RAW_DATASET / "dataset.json", "w"), indent=2)
shutil.copy(SRC / "splits_final.json", RAW_DATASET / "splits_final.json")
REPORT["file_ending_used"] = ending
print("nnU-Net raw dataset:", RAW_DATASET, "| file ending:", ending)

In [ ]:
!pip install -q nnunetv2 2>&1 | tail -2
REPORT["env"]["nnunetv2"] = subprocess.getoutput("pip show nnunetv2 | grep Version")
print(REPORT["env"]["nnunetv2"])

In [ ]:
# 2. nnU-Net folders, planning and preprocessing (once, shared by all runs)
NN = Path("/tmp/nnunet")
os.environ["nnUNet_raw"] = str(RAW_DATASET.parent)      # read-only input is fine: raw data is only read
os.environ["nnUNet_preprocessed"] = str(NN / "preprocessed")
os.environ["nnUNet_results"] = str(NN / "results")
for k in ("nnUNet_preprocessed", "nnUNet_results"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

t0 = time.time()
log = subprocess.run("nnUNetv2_plan_and_preprocess -d 1 -c 3d_fullres -np 4 --verify_dataset_integrity", shell=True, capture_output=True, text=True)
print(log.stdout[-1500:], log.stderr[-1500:])
assert log.returncode == 0, "planning/preprocessing failed"
PRE = Path(os.environ["nnUNet_preprocessed"]) / "Dataset001_WMH"
shutil.copy(RAW_DATASET / "splits_final.json", PRE / "splits_final.json")   # our stratified folds (J-032)
plans = json.load(open(PRE / "nnUNetPlans.json"))["configurations"]["3d_fullres"]
REPORT["preprocess_s"] = round(time.time() - t0, 1)
REPORT["plan"] = {k: plans.get(k) for k in ("patch_size", "batch_size", "spacing")}
print(REPORT["preprocess_s"], "s |", REPORT["plan"])

In [ ]:
# 3. One timed run: train fold 0, stop after N measured epochs, sample GPU and CPU usage
def gpu_util():
    out = subprocess.getoutput("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader,nounits")
    try:
        return [tuple(float(x) for x in line.split(",")) for line in out.strip().splitlines()]
    except ValueError:
        return []

def kill_tree(pid):
    try:
        parent = psutil.Process(pid)
        for p in parent.children(recursive=True) + [parent]:
            p.kill()
    except psutil.Error:
        pass

def timed_run(name, n_proc_da, compile_, num_gpus, n_epochs=4, timeout_s=1800):
    out_dir = Path(os.environ["nnUNet_results"]) / "Dataset001_WMH"
    shutil.rmtree(out_dir, ignore_errors=True)   # fresh start for each run
    env = dict(os.environ, nnUNet_n_proc_DA=str(n_proc_da), nnUNet_compile="t" if compile_ else "f")
    cmd = f"nnUNetv2_train 1 3d_fullres 0 -tr nnUNetTrainer_250epochs -num_gpus {num_gpus}"
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, env=env, stdout=open(f"/tmp/{name}.log", "w"), stderr=subprocess.STDOUT)
    samples, epochs, logs = [], [], []
    while proc.poll() is None and time.time() - t0 < timeout_s:
        time.sleep(3)
        g = gpu_util()
        samples.append({"t": time.time() - t0, "gpu": [u for u, _ in g], "vram_gb": [m / 1024 for _, m in g],
                        "cpu": psutil.cpu_percent(interval=None), "n_epochs_done": len(epochs)})
        logs = glob.glob(str(out_dir / "**/training_log_*.txt"), recursive=True)
        if logs:
            epochs = [float(x) for x in re.findall(r"Epoch time: ([\d.]+) s", open(logs[0]).read())]
        if len(epochs) >= n_epochs:
            break
    kill_tree(proc.pid)
    # utilisation measured after the first (compilation / warm-up) epoch only
    steady = [s for s in samples if s["n_epochs_done"] >= 1 and s["gpu"]]
    mean = lambda xs: round(sum(xs) / len(xs), 1) if xs else None
    res = {"name": name, "n_proc_DA": n_proc_da, "compile": compile_, "num_gpus": num_gpus,
           "epoch_times_s": epochs, "median_s_excl_first": sorted(epochs[1:])[len(epochs[1:]) // 2] if len(epochs) > 1 else None,
           "first_epoch_s": epochs[0] if epochs else None, "wall_s": round(time.time() - t0, 1),
           "gpu_util_mean_pct": [mean([s["gpu"][i] for s in steady if len(s["gpu"]) > i]) for i in range(torch.cuda.device_count())],
           "vram_max_gb": [round(max([s["vram_gb"][i] for s in steady if len(s["vram_gb"]) > i] or [0]), 1) for i in range(torch.cuda.device_count())],
           "cpu_util_mean_pct": mean([s["cpu"] for s in steady])}
    if not epochs:
        res["log_tail"] = open(f"/tmp/{name}.log").read()[-1500:]
    print(json.dumps(res, indent=1))
    return res

REPORT["runs"] = []

In [ ]:
REPORT["runs"].append(timed_run("A_1gpu_da3_compile", n_proc_da=3, compile_=True, num_gpus=1))

In [ ]:
REPORT["runs"].append(timed_run("B_1gpu_da4_compile", n_proc_da=4, compile_=True, num_gpus=1))

In [ ]:
REPORT["runs"].append(timed_run("C_1gpu_da4_nocompile", n_proc_da=4, compile_=False, num_gpus=1))

In [ ]:
if torch.cuda.device_count() >= 2:
    REPORT["runs"].append(timed_run("D_2gpu_da4_compile", n_proc_da=4, compile_=True, num_gpus=2))
else:
    print("Only one GPU: run D skipped (choose GPU T4 x2 in Settings)")

In [ ]:
# 4. Summary: projected training time (epochs x median epoch time + ~10 min final validation of 12 cases)
rows = []
for r in REPORT["runs"]:
    m = r["median_s_excl_first"]
    if m is None:
        continue
    proj = {n: round((r["first_epoch_s"] + (n - 1) * m) / 3600 + 10 / 60, 1) for n in (100, 150, 200, 250)}
    rows.append({"run": r["name"], "s_per_epoch": m, "gpu_util": r["gpu_util_mean_pct"], "cpu_util": r["cpu_util_mean_pct"],
                 **{f"h_{n}ep": h for n, h in proj.items()}})
REPORT["summary"] = rows
REPORT["notes"] = {"kaggle_session_limit_h": 12, "projection": "first epoch + (n-1) x median, + ~10 min final validation (estimate)"}
for row in rows:
    print(row)
REPORT["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
json.dump(REPORT, open("/kaggle/working/epoch_benchmark.json", "w"), indent=1)
print("\n=== Copier ce bloc dans la conversation ===")
print(json.dumps(REPORT, indent=1))